# 04 — Evaluation

Loads a trained model and evaluates it on the **held-out test split**
(never seen during training or hyperparameter tuning), reporting:

- Accuracy, precision, recall, F1 (overall + per-class) — matching the
  metrics reported in the base paper.
- Confusion matrix (raw counts + normalized).
- One-vs-rest ROC curves + AUC per class.
- A saved predictions CSV for auditability.

Run `03_training.ipynb` first so `models/<backbone>_final_model.keras`
exists.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import Image, display

from src.preprocessing.video_loader import build_dataset
from src.evaluation.metrics import compute_metrics, metrics_to_percent_string
from src.evaluation.confusion_matrix import compute_confusion_matrix, plot_confusion_matrix
from src.evaluation.roc_curve import plot_multiclass_roc
from src.utils.config import DEFAULT_CONFIG, MODELS_DIR, OUTPUTS_DIR

%matplotlib inline

## Choose which trained backbone to evaluate

In [ ]:
cfg = DEFAULT_CONFIG
cfg.backbone = "mobilenet"   # must match a backbone you already trained in 03_training.ipynb

model_path = os.path.join(MODELS_DIR, f"{cfg.backbone}_final_model.keras")
print("Loading:", model_path)
model = tf.keras.models.load_model(model_path)

## Load the test split and run inference

In [ ]:
test_ds, n_test = build_dataset("test", cfg, shuffle=False)
print(f"Evaluating on {n_test} test samples.")

y_true, y_pred_proba = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_proba.append(preds)
    y_true.append(np.argmax(labels.numpy(), axis=1))

y_true = np.concatenate(y_true)
y_pred_proba = np.concatenate(y_pred_proba)
y_pred = np.argmax(y_pred_proba, axis=1)

## Overall + per-class metrics

In [ ]:
metrics = compute_metrics(y_true, y_pred, cfg.class_names)
print(metrics_to_percent_string(metrics))
print()
print(metrics["report_text"])
metrics["per_class"]

Pay particular attention to **PD recall** here: because PD has the fewest
videos in the base dataset, a model can post a respectable overall
accuracy while still missing most actual PD cases. If PD recall is
noticeably lower than KOA/Healthy recall, that's the imbalance showing up,
not necessarily a bug -- see the "Improving accuracy" notes in the project
README for mitigation options (class weighting, oversampling, more data).

## Confusion matrix

In [ ]:
cm = compute_confusion_matrix(y_true, y_pred, cfg.class_names)
print(cm)

counts_path = plot_confusion_matrix(cm, cfg.class_names, cfg.backbone, normalize=False)
norm_path = plot_confusion_matrix(cm, cfg.class_names, cfg.backbone, normalize=True)

display(Image(filename=counts_path))
display(Image(filename=norm_path))

## ROC curves (one-vs-rest) + AUC per class

In [ ]:
roc_path = plot_multiclass_roc(y_true, y_pred_proba, cfg.class_names, cfg.backbone)
display(Image(filename=roc_path))

## Save per-sample predictions for auditability

In [ ]:
predictions_dir = os.path.join(OUTPUTS_DIR, "predictions")
os.makedirs(predictions_dir, exist_ok=True)

pred_df = pd.DataFrame({
    "true_label": [cfg.class_names[i] for i in y_true],
    "predicted_label": [cfg.class_names[i] for i in y_pred],
    **{f"prob_{name}": y_pred_proba[:, i] for i, name in enumerate(cfg.class_names)},
})

out_csv = os.path.join(predictions_dir, f"{cfg.backbone}_test_predictions.csv")
pred_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
pred_df.head(10)

## Optional: compare multiple backbones side by side

Re-run this notebook once per backbone (`vgg16`, `mobilenet`,
`densenet`), collecting `metrics['accuracy']` etc. each time, then build a
small comparison table like Table 8 in the base paper. Example scaffold
if you've already run all three and saved predictions CSVs:

In [ ]:
# backbones = ["vgg16", "mobilenet", "densenet"]
# rows = []
# for b in backbones:
#     df_b = pd.read_csv(os.path.join(OUTPUTS_DIR, "predictions", f"{b}_test_predictions.csv"))
#     y_t = df_b["true_label"].map({n: i for i, n in enumerate(cfg.class_names)}).values
#     y_p = df_b["predicted_label"].map({n: i for i, n in enumerate(cfg.class_names)}).values
#     m = compute_metrics(y_t, y_p, cfg.class_names)
#     rows.append({"backbone": b, **{k: m[k] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]}})
# pd.DataFrame(rows)